# TensorScript → T4 (Colab)

Run the full SFT + DPO pipeline from the TensorScript v1.0 spec, compiled to
real `transformers`/`peft` code, on a Colab T4 (16 GB). Uses QLoRA (4-bit base),
fp16, gradient checkpointing — the `hardware: {profile: "t4"}` block drives all
of that automatically.

**Runtime:** T4 GPU · High-RAM


In [ ]:
# Cell 1 — deps (run once)
!pip -q install -U transformers peft accelerate datasets trl bitsandbytes wandb
!nvidia-smi


## Cell 2 — spec

This is the TensorScript spec. The transpiler turns it into Python you could
write by hand — but with the guardrails, telemetry, pipeline wiring, and
hardware profile baked in.


In [ ]:
%%writefile align_example.tensor
tensorscript v1.0

dataset SFTData streams {
    source: "hf://datasets/HuggingFaceH4/ultrachat_200k",
    tokenize: AutoTokenizer("meta-llama/Llama-3-8B"),
    sequence_length: 1024,
    split: [train_sft: 99%, val_sft: 1%]
}

model BaseLM {
    base: "hf://meta-llama/Meta-Llama-3-8B",
    quantize: 4bit,
    peft: LoRA(r=16, alpha=32, target_modules="all-linear")
}

monitor RunTelemetry {
    track: [loss, perplexity, memory_usage, reward_margin, kl_divergence],
    checkpoint_every: 500.steps,
    select_best_on: loss minimize,
    destination: "wandb://pioneer-space/sft-dpo-pipeline"
}

optimize BaseLM as SFTStage {
    using: dataset.SFTData.train_sft,
    hardware: {profile: "t4"},
    telemetry: monitor.RunTelemetry,
    schedule: CosineAnnealing(max_lr=2e-4, min_lr=2e-5, warmup=0.03),
    epochs: 1,
    guardrails: [
        if loss > 2.5: rollback_and_scale_lr(0.5),
        if loss flat_lines(epsilon=1e-4) for 200.steps: terminate_early
    ]
}

dataset PreferenceData streams {
    source: "hf://datasets/Anthropic/hh-rlhf",
    tokenize: AutoTokenizer("meta-llama/Llama-3-8B"),
    sequence_length: 1024,
    split: [train: 90%, val: 10%]
}

model DPOModel {
    base: pipeline.SFTStage.best,
    quantize: 4bit,
    peft: LoRA(r=16, alpha=32, target_modules="all-linear")
}

optimize DPOModel as DPOStage {
    using: dataset.PreferenceData.train,
    hardware: {profile: "t4"},
    telemetry: monitor.RunTelemetry,
    schedule: CosineAnnealing(max_lr=5e-6, min_lr=5e-7, warmup=0.1),
    epochs: 1,
    guardrails: [
        if reward_margin < 0: rollback_and_scale_lr(0.3),
        if kl_divergence > 10: rollback_and_scale_lr(0.5)
    ]
}

pipeline AlignmentRun {
    stage one {
        run: optimize.SFTStage
    },
    stage two {
        run: optimize.DPOStage,
        depends_on: one,
        inherit_weights: best
    }
}


## Cell 3 — transpile (auto-downloads the toolchain)

The toolchain is fetched straight from your GitHub repo, then validated, then
the transpiler emits `SFTStage.py`, `DPOStage.py`, and `run_AlignmentRun.py`.
No manual uploads needed.


In [ ]:
# Cell 3 — fetch toolchain + validate + transpile
import sys, os, py_compile

BASE = "https://raw.githubusercontent.com/sanctuary-architect/Tensorscript/main/"
files_to_download = ["parser.py", "transpiler.py", "semantic_checks.py", "tsc_check.py", "lexer.py"]

print("Ensuring toolchain files are present...")
for f in files_to_download:
    if not os.path.exists(f):
        print(f"Downloading {f}...")
        !wget -q $BASE$f

for f in ["parser.py", "transpiler.py", "semantic_checks.py", "lexer.py"]:
    py_compile.compile(f, doraise=True)
print("Toolchain ready")

import sys; sys.path.insert(0, ".")
from semantic_checks import check
from parser import parse
from transpiler import transpile

ast = parse(open("align_example.tensor").read())
check(ast)
out = transpile(ast, out_dir=".")

# Small alignment fixes for the generated stages (matching the spec):
for filename, content in out.items():
    if filename in ["SFTStage.py", "DPOStage.py"]:
        lines = content.splitlines()
        new_lines = []
        for line in lines:
            if "save_steps=" in line:
                line = "    save_steps=500,"
            if "eval_steps=" in line:
                line = "    eval_steps=500,"
            if "split='train'" in line and "ultrachat_200k" in content:
                line = line.replace("split='train'", "split='train_sft'")
            new_lines.append(line)
        with open(filename, "w") as f:
            f.write("\n".join(new_lines))
        print(f"Successfully wrote {filename}")
    else:
        with open(filename, "w") as f:
            f.write(content)
        print(f"Successfully wrote {filename}")
print("Generated:", os.listdir('.'))


## Cell 4 — Hugging Face login

`meta-llama/Meta-Llama-3-8B` is a **gated model**: you must accept its license on
huggingface.co (Meta's terms) and paste a read token here.

🔐 **Private-key safety:** this box runs a login *prompt* — your real token is
typed into it at runtime and is **never saved into this notebook file**, so it
can't leak if the notebook is public. Create a read-only token at
huggingface.co/settings/tokens (type: Read).


In [ ]:
# Cell 4 — log in to Hugging Face (paste your READ token below)
from huggingface_hub import notebook_login
notebook_login()


## Cell 5 — run the pipeline

SFT first (download + fine-tune), then DPO on the SFT best checkpoint.
`run_AlignmentRun.py` resolves `pipeline.SFTStage.best` automatically.

> ⚠️ `sequence_length: 1024` + batch 1 + gradient checkpointing ≈ 12–14 GB
> VRAM. If you hit OOM, drop to `sequence_length: 512`.


In [ ]:
# Cell 5 — execute (SFT then DPO; ~30–60 min on a T4)
!python run_AlignmentRun.py


## Cell 6 — inspect the guardrail log + outputs

The console shows `[guardrail]` warnings if a rule fired. The outputs land in
`./output/SFTStage` and `./output/DPOStage`.


In [ ]:
# Cell 6 — inspect outputs
import glob, os
for d in ["output/SFTStage", "output/DPOStage"]:
    if os.path.isdir(d):
        print(d, "->", os.listdir(d)[:6])

with open('SFTStage.py', 'r') as f:
    lines = f.readlines()
    for i, line in enumerate(lines):
        if 'TrainingArguments(' in line:
            print("".join(lines[i:i+20]))
            break
